# Day 4-03｜球的追蹤與出手點估計

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：先用簡單顏色追蹤理解球軌跡，再討論真實影片中會遇到的遮擋、光線、球太小等問題。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
from src.video_utils import pick_first_converted_video
from src.shooting_utils import track_orange_ball, estimate_release_frame
from src.plot_utils import plot_ball_path

# 若學生沒有上傳影片，會自動使用 assets/samples/sample_ball_motion.mp4。
video_path = pick_first_converted_video(COURSE_ROOT)
print("video_path:", video_path)

In [ ]:
ball_df = track_orange_ball(video_path, max_frames=180)
print("tracked points:", len(ball_df))
ball_df.head()

In [ ]:
out_csv = COURSE_ROOT / "assets" / "results" / "d4_03_ball_track.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
ball_df.to_csv(out_csv, index=False)
print("saved:", out_csv)

plot_ball_path(
    ball_df, output_path=COURSE_ROOT / "assets" / "results" / "d4_03_ball_path.png"
)
release_frame = estimate_release_frame(ball_df)
print("estimated release frame:", release_frame)

## 討論

簡單顏色追蹤在下列情況會失敗：

- 球被手擋住。
- 球跟背景顏色太接近。
- 室外光線變化大。
- 影片壓縮太嚴重。

進階做法可以改成：YOLO ball detector、手動標幾個 frame、或用 tracking + interpolation。